#### Autores:
- Iker Alcaide Velasco, 100498961
- Alejandra Cardó González, 100499061

# Predicciones de Datos para Competición

Importamos las librerías necesarias para la manipulación de los datos y la carga del modelo:

In [25]:
import pickle
import pandas as pd
import numpy as np

Cargamos en memoria el modelo final que hemos entrenado y serializado previamente:

In [26]:

with open("modelo_final.pkl", "rb") as f:
    modelo_final = pickle.load(f)

Cargamos el conjunto de datos de evaluación (competición) sobre el cual queremos generar las nuevas predicciones:

In [27]:

with open("bank_competition_61.pkl", "rb") as f:
    df_comp = pickle.load(f)

/tmp/ipykernel_640/105424484.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df_comp = pickle.load(f)


Generamos las predicciones utilizando nuestro modelo cargado y las exportamos a un archivo `.csv` para su posterior entrega. Finalmente, comprobamos la distribución de las etiquetas predichas:

In [28]:
predicciones = modelo_final.predict(df_comp)


pred_df = pd.DataFrame({"deposit": predicciones})
pred_df.to_csv("predicciones.csv", index=False)
print("Predicciones guardadas en 'predicciones.csv'")
print(pred_df["deposit"].value_counts())

Predicciones guardadas en 'predicciones.csv'
deposit
no     91
yes    71
Name: count, dtype: int64


# Predicciones para demostración de Streamlit

Creamos dos instancias nuevas de ejemplo, que no se incluirán en `predicciones.csv`, una con características propensas a NO contrartar y otra con características propensas a SÍ contratar.

#### Cliente 1: Propenso a no contratar

In [29]:
cliente_1 = {
    "age": 40,
    "balance": 0,
    "day": 15,
    "duration": 200,
    "campaign": 1,
    "previous": 0,
    "pdays": -1,
    "month": "may",
    "job": "admin.",
    "marital": "married",
    "education": "secondary",
    "default": "no",
    "housing": "yes",
    "loan": "no",
    "contact": "unknown",
    "poutcome": "unknown"
}

#### Cliente 2: Propenso a SÍ contratar
Mayor  de llamada, éxito previo, etc..

In [30]:
cliente_2 = {
    "age": 60,
    "balance": 5100,
    "day": 10,
    "duration": 800,
    "campaign": 2,
    "previous": 3,
    "pdays": 90,
    "month": "mar",
    "job": "retired",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "housing": "no",
    "loan": "no",
    "contact": "cellular",
    "poutcome": "success"
}

Hacemos un data frame nuevo:

In [31]:
df_nuevos = pd.DataFrame([cliente_1, cliente_2])

#### Predecimos con el pipeline

In [32]:
predicciones_nuevas = modelo_final.predict(df_nuevos)

Manejamos si tiene probabilidad el modelo, SVM no la da pero HGB sí.

In [33]:
has_proba = hasattr(modelo_final, "predict_proba")
if has_proba:
    probabilidades = modelo_final.predict_proba(df_nuevos)
    clases = list(modelo_final.classes_)

#### Imprimimos los resultados

Imprimimmos **datos de los clientes**:

In [34]:
print("\n[Cliente 1 - Perfil negativo]")
print(cliente_1)
print("\n[Cliente 2 - Perfil positivo]")
print(cliente_2)


[Cliente 1 - Perfil negativo]
{'age': 40, 'balance': 0, 'day': 15, 'duration': 200, 'campaign': 1, 'previous': 0, 'pdays': -1, 'month': 'may', 'job': 'admin.', 'marital': 'married', 'education': 'secondary', 'default': 'no', 'housing': 'yes', 'loan': 'no', 'contact': 'unknown', 'poutcome': 'unknown'}

[Cliente 2 - Perfil positivo]
{'age': 60, 'balance': 5100, 'day': 10, 'duration': 800, 'campaign': 2, 'previous': 3, 'pdays': 90, 'month': 'mar', 'job': 'retired', 'marital': 'married', 'education': 'tertiary', 'default': 'no', 'housing': 'no', 'loan': 'no', 'contact': 'cellular', 'poutcome': 'success'}


Imprimimos **resultado del pipeline**:

In [35]:
print("Predicción Cliente 1:", predicciones[0].upper())
print("Predicción Cliente 2:", predicciones[1].upper())

Predicción Cliente 1: NO
Predicción Cliente 2: YES


Imprimimos **probabilidades** (de tenerlas):

In [36]:
if has_proba:
    print("\nPROBABILIDADES:")
    for i, prob in enumerate(probabilidades):
        prob_str = ", ".join([f"{clases[j]}: {prob[j]:.4f}" for j in range(len(clases))])
        print(f"Cliente {i+1}: {prob_str}")